## 필요한 라이브러리 다운하기

In [1]:
!pip install yt-dlp

   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   --------------- ------------------------ 1.3/3.3 MB 6.6 MB/s eta 0:00:01
   ------------------------------- -------- 2.6/3.3 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 3.3/3.3 MB 6.6 MB/s eta 0:00:00


In [3]:
!pip install mysql-connector-python

  Using cached mysql_connector_python-9.5.0-cp313-cp313-win_amd64.whl.metadata (7.7 kB)
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
    --------------------------------------- 0.3/16.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.5 MB 2.0 MB/s eta 0:00:09
   --- ------------------------------------ 1.3/16.5 MB 2.1 MB/s eta 0:00:08
   ---- ----------------------------------- 1.8/16.5 MB 2.2 MB/s eta 0:00:07
   ----- ---------------------------------- 2.4/16.5 MB 2.2 MB/s eta 0:00:07
   ------ --------------------------------- 2.6/16.5 MB 2.1 MB/s eta 0:00:07
   ------- -------------------------------- 3.1/16.5 MB 2.1 MB/s eta 0:00:07
   -------- ------------------------------- 3.4/16.5 MB 2.1 MB/s eta 0:00:07
   -------- ------------------------------- 3.7/16.5 MB 2.0 MB/s eta 0:00:07
   -------- ------------------------------- 3.7/16.5 MB 2.0 MB/s eta 0:00:07
   ------

In [3]:
!pip install pymysql

## 데이터 수집을 위한 프로그래밍


In [1]:
import yt_dlp
import mysql.connector
import time
import re

import warnings
warnings.filterwarnings('ignore')

from mysql.connector import Error

In [2]:
try:
    conn = mysql.connector.connect(
        host='127.0.0.1',
        user='inho123',
        password='a3409440@',
        database='youtube_recipe'
    )
    if conn.is_connected():
        print('데이터베이스 연결에 성공했습니다')
    conn.close()
except Exception as e:
    print(f'연결 오류 발생 사유: {e}')

데이터베이스 연결에 성공했습니다


In [ ]:
class REF_Youtube_Recipe_Collector:
	def __init__(self, db_config):
		# 수집 설정
		self.ydl_opts = {
			'getcomments': True, # 댓글 허용
			'skip_download': True, # 영상 다운 스킵
			'no_warnings': True, # 경고 무시
			'ignoreerrors': True, # 에러나도 계속 
			'nocheckcertificate': True, # 인증서 스킵
			'quiet': True, # 과정 보여주기 X
			'logger': None, # yt-dlp 자체 로거를 꺼버려서 내부 에러 출력을 차단

            'playlist_items': '497-', # 143번쨰 영상부터 크롤링 설정
			
			# 목록을 기다리지 않고 발견 즉시 상세 페이지로 접속하기 위한 설정
			'extract_flat': 'in_playlist', 
			'lazy_playlist': True, # 목록 로딩을 스트리밍 방식으로 처리
			
			'extractor_args': {'youtube': {'player_client': ['android', 'web']}}, # 차단 우회 보강
			'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
			'referer': 'https://www.google.com/',
			'language': 'ko', # 유튜브 서버에 한국어 데이터를 우선적으로 요청
		}
		self.db_config = db_config # 추후 DB에 접속할 떄 쓰는 주소와 비번
		# 레시피인지 판단할 키워드들
		self.recipe_keyword_list = ['큰술', '스푼', 'g', 'ml', '재료', '조리법', '만드는 법', '적당량', '1/2', '1/3', '2/3',
						'만드는 법', '만드는법']

	# 레시피 판별 로직
	def check_is_recipe(self, target_text): 
		# 텍스트에 요리 관련 키워드 있는지 검사. 있으면 True, 없으면 False
		if not target_text: return False

		# 한글이 최소 1글자라도 들어있는지 검사 (영어/외국어 전용 레시피 차단)
		is_korean_included = bool(re.search('[가-힣]', target_text))
		if not is_korean_included:
			return False

		return any(keyword in target_text for keyword in self.recipe_keyword_list)

	# 메인 수집 로직
	def collect_and_save_recipe_list(self, video_url_list):
		# 영상 리스트를 순회하며 수집 및 DB 저장
		with yt_dlp.YoutubeDL(self.ydl_opts) as ydl_instance:
			for video_url in video_url_list: # 입력받는 유튜브 주소 받아서 반복
				print(f"분석 시작 (실시간 상세 모드): {video_url}")
				try:
					# 목록만 먼저 빠르게 스트리밍으로 가져옴
					video_info_dict = ydl_instance.extract_info(video_url, download=False, process=False) 
					
					if 'entries' in video_info_dict:
						current_num = 497
						for entry in video_info_dict['entries']:
							# 목록의 각 항목에 대해 개별적으로 상세 정보(전체 설명란)를 추출
							try:
								entry_url = entry.get('url') or entry.get('webpage_url')
								if not entry_url: continue
								
								video_entry = ydl_instance.extract_info(entry_url, download=False)
							except:
								continue
								
							if not video_entry: continue
							
							# 실제 영상 데이터를 한 번 더 확실히 가져옴
							video_title_name = video_entry.get('title', '제목 없음')
							current_video_url = video_entry.get('webpage_url') or video_entry.get('url')
							
							# 실시간 카운팅 출력 (이모지 제거)
							print(f"[{current_num}번째 영상] 분석 중: {video_title_name}")
							
							# shorts 구분 로직
							video_duration_seconds = video_entry.get('duration', 0)
							video_type_name = 'SHORTS' if video_duration_seconds <= 60 else 'VIDEO'
							
							# 데이터 정리(설명란 + 댓글)된 것들을 저장
							recipe_stock_data_list = []
							
							# 설명란 체크하고 저장 
							# 상세 수집 모드이므로 description에 전체 원문이 온전히 담김
							video_description_content = video_entry.get('description', '')
							
							# 설명란도 키워드 검사를 거쳐 레시피인 경우에만 저장
							if self.check_is_recipe(video_description_content):
								recipe_stock_data_list.append(('DESCRIPTION', video_description_content))
							
							# 댓글 체크
							video_comment_list = video_entry.get('comments', []) or [] # 댓글 하나하나 쪼개서 검사 
							for comment_item in video_comment_list:
								comment_text_content = comment_item.get('text', '')
								if self.check_is_recipe(comment_text_content): # 레시피라고 판단되면
									recipe_stock_data_list.append(('COMMENT', comment_text_content)) # 담음 

							# 3. DB 접속 및 저장
							if recipe_stock_data_list:
								self.insert_recipe_stock_to_db(video_title_name, current_video_url, video_type_name, recipe_stock_data_list)
							else:
								print(f"건너뜀: {video_title_name} (레시피 키워드 없음)")
							
							current_num += 1
							time.sleep(1) # 차단 방지를 위한 최소한의 매너 타임

				except Exception as error_instance:
					print(f"오류 발생 ({video_url}): {error_instance}") # 오류 발생시 안 멈추도록 장치

	# DB 저장 로직
	def insert_recipe_stock_to_db(self, video_title_name, video_url, video_type_name, recipe_stock_data_list):
		# MySQL 저장 로직 (중복은 무시)
		db_connection = None # 통로를 담을 변수 
		try:
			db_connection = mysql.connector.connect(**self.db_config) # 아까 설정한 비번, 주소로 DB접속
			db_cursor = db_connection.cursor() # 데이터 하나씩 집어넣어주는 도구 
			
			# %s를 5개로 맞추어 DB 컬럼 개수와 일치시킴
			sql_query = """
				INSERT IGNORE INTO recipes (video_title, video_url, video_type, source_type, recipe_content) 
				VALUES (%s, %s, %s, %s, %s)
			""" # sql recipe 테이블에 넣는데 중복이면 무시하라는 명령 
			
			inserted_count = 0 # 새로 저장된 데이터 개수를 셀 카운트
			for source_type_name, ref_fridge_stock_content in recipe_stock_data_list: # 담아놓은 데이터 꺼내 DB에 하나씩 넣기
				db_cursor.execute(sql_query, (video_title_name, video_url, video_type_name, source_type_name, ref_fridge_stock_content))
				if db_cursor.rowcount > 0: # 한 줄 저장되면 카운트
					inserted_count += 1
			
			db_connection.commit() # 이게 있어야 영구적인 저장이 가능함!!!
			
			if inserted_count > 0:
				print(f"결과: {inserted_count}건의 새로운 레시피 저장 완료")
				
		except mysql.connector.Error as db_error: # 비번이나 주소 잘못되면 에러 내용 알려주기
			print(f"DB 오류 발생: {db_error}")
		finally:
			if db_connection and db_connection.is_connected():
				db_cursor.close()
				db_connection.close()

if __name__ == "__main__":
	my_db_settings = {
		'host': '127.0.0.1',
		'user': 'inho123',
		'password': 'a3409440@', 
		'database': 'youtube_recipe'
	}

	target_video_url_list = [#"https://www.youtube.com/@자취요리신simplecooking/videos", #자취요리
                             #"https://www.youtube.com/@-yumyumeasymeal/videos", #간단요리
                             "https://www.youtube.com/@만개의레시피/videos", #만개의 레시피
                             #"https://www.youtube.com/@ddubi_doobab/shorts", #뚜비두밥 맛있는 밥
                            ] 

	ref_recipe_collector = REF_Youtube_Recipe_Collector(my_db_settings)
	ref_recipe_collector.collect_and_save_recipe_list(target_video_url_list)

In [23]:
import yt_dlp
import mysql.connector
import time
import re
import random # 랜덤 휴식 시간을 위한 모듈

class REF_Youtube_Recipe_Collector:
	def __init__(self, db_config):
		# 수집 설정
		self.ydl_opts = {
			'getcomments': True, # 댓글 허용
			'skip_download': True, # 영상 다운 스킵
			'no_warnings': True, # 경고 무시
			'ignoreerrors': True, # 에러나도 계속 
			'nocheckcertificate': True, # 인증서 스킵
			'quiet': True, # 과정 보여주기 X
			'logger': None, # yt-dlp 자체 로거를 꺼버려서 내부 에러 출력을 차단
			
			# [보강] playlist_items를 명시적으로 고정
			'playlist_items': '498-', 
			
			'extract_flat': 'in_playlist', 
			'lazy_playlist': True, 
			'extractor_args': {'youtube': {'player_client': ['android', 'web']}}, 
			'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
			'referer': 'https://www.google.com/',
			'language': 'ko', 
		}
		self.db_config = db_config # 추후 DB에 접속할 떄 쓰는 주소와 비번
		# 레시피인지 판단할 키워드들
		self.recipe_keyword_list = ['큰술', '스푼', 'g', 'ml', '재료', '조리법', '만드는 법', '적당량', '1/2', '1/3', '2/3',
						'만드는 법', '만드는법']

	# 레시피 판별 로직
	def check_is_recipe(self, target_text): 
		# 텍스트에 요리 관련 키워드 있는지 검사. 있으면 True, 없으면 False
		if not target_text: return False
		is_korean_included = bool(re.search('[가-힣]', target_text))
		if not is_korean_included:
			return False
		return any(keyword in target_text for keyword in self.recipe_keyword_list)

	# 메인 수집 로직
	def collect_and_save_recipe_list(self, video_url_list):
		# 영상 리스트를 순회하며 수집 및 DB 저장
		with yt_dlp.YoutubeDL(self.ydl_opts) as ydl_instance:
			for video_url in video_url_list: # 입력받는 유튜브 주소 받아서 반복
				print(f"분석 시작 (지정 구간 498번부터 강제 실행): {video_url}")
				try:
					# [보강] playlist_items 설정을 확실히 적용하기 위해 extract_info 재설정
					video_info_dict = ydl_instance.extract_info(video_url, download=False, process=False) 
					
					if 'entries' in video_info_dict:
						# [수정] 수집 시작 번호를 498로 고정
						current_num = 498 
						
						for entry in video_info_dict['entries']:
							# [보강] 만약 내부 인덱싱 문제로 1번부터 가져온다면 강제로 건너뛰는 안전장치
							entry_url = entry.get('url') or entry.get('webpage_url')
							if not entry_url: continue
							
							try:
								video_entry = ydl_instance.extract_info(entry_url, download=False)
							except Exception as e:
								print(f"접근 실패: {entry_url}")
								continue
								
							if not video_entry: continue
							
							video_title_name = video_entry.get('title', '제목 없음')
							current_video_url = video_entry.get('webpage_url') or video_entry.get('url')
							
							print(f"[{current_num}번째 영상] 분석 중: {video_title_name}")
							
							video_duration_seconds = video_entry.get('duration', 0)
							video_type_name = 'SHORTS' if video_duration_seconds <= 60 else 'VIDEO'
							
							recipe_stock_data_list = []
							video_description_content = video_entry.get('description', '')
							
							if self.check_is_recipe(video_description_content):
								recipe_stock_data_list.append(('DESCRIPTION', video_description_content))
							
							video_comment_list = video_entry.get('comments', []) or [] 
							for comment_item in video_comment_list:
								comment_text_content = comment_item.get('text', '')
								if self.check_is_recipe(comment_text_content): 
									recipe_stock_data_list.append(('COMMENT', comment_text_content))

							# 3. DB 접속 및 저장
							if recipe_stock_data_list:
								self.insert_recipe_stock_to_db(video_title_name, current_video_url, video_type_name, recipe_stock_data_list)
							else:
								print(f"건너뜀: {video_title_name} (레시피 키워드 없음)")
							
							current_num += 1
							# 차단 방지를 위해 불규칙한 휴식 시간을 넉넉히 부여
							time.sleep(random.uniform(5.0, 10.0)) 

				except Exception as error_instance:
					print(f"오류 발생 ({video_url}): {error_instance}")

	# DB 저장 로직 (원본 주석 유지)
	def insert_recipe_stock_to_db(self, video_title_name, video_url, video_type_name, recipe_stock_data_list):
		# MySQL 저장 로직 (중복은 무시)
		db_connection = None # 통로를 담을 변수 
		try:
			db_connection = mysql.connector.connect(**self.db_config) # DB접속
			db_cursor = db_connection.cursor() 
			sql_query = """
				INSERT IGNORE INTO recipes (video_title, video_url, video_type, source_type, recipe_content) 
				VALUES (%s, %s, %s, %s, %s)
			""" 
			inserted_count = 0 
			for source_type_name, ref_fridge_stock_content in recipe_stock_data_list:
				db_cursor.execute(sql_query, (video_title_name, video_url, video_type_name, source_type_name, ref_fridge_stock_content))
				if db_cursor.rowcount > 0:
					inserted_count += 1
			db_connection.commit() 
			if inserted_count > 0:
				print(f"결과: {inserted_count}건의 새로운 레시피 저장 완료")
		except mysql.connector.Error as db_error:
			print(f"DB 오류 발생: {db_error}")
		finally:
			if db_connection and db_connection.is_connected():
				db_cursor.close()
				db_connection.close()